In [1]:
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

In [2]:
data_dir="/kaggle/input/datasets/ayush1220/cifar10/cifar10"
train_transform=transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )
])

test_transform=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )
])

In [3]:
trainset = ImageFolder(root=f"{data_dir}/train", transform=train_transform)
trainloader = DataLoader(trainset, batch_size=32, shuffle=True)

testset = ImageFolder(root=f"{data_dir}/test", transform=test_transform)
testloader = DataLoader(testset, batch_size=32, shuffle=False)

In [27]:
from tqdm import tqdm
import torch.optim as optim

def train_model(model,trainloader,epochs):
    model.train()
    optimizer=optim.Adam(model.parameters(),lr=1e-4)
    criterion=nn.CrossEntropyLoss()
    total=len(trainloader)
    for epoch in range(epochs):
        running_loss=0
        progress_bar=tqdm(trainloader)
        for inputs, labels in progress_bar :
    
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss+=loss.item()
    
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            progress_bar.set_description(f"epoch: {epoch+1}")
            progress_bar.set_postfix(loss=loss.item())
        print(f"epoch:{epoch+1}  loss:{running_loss/total}")
        

In [18]:
def eval_model(model,test_loader):
    model.eval()
    correct=0
    total=0
    for images, labels in tqdm(testloader):
        outputs=model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    return (correct/total)*100

In [9]:
# Custom CNN Model
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.features=nn.Sequential(
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier=nn.Sequential(
            nn.Flatten(),
            nn.Linear(4*4*128,256),
            nn.ReLU(),
            nn.Linear(256,10)
        )

    def forward(self,x):
        x=self.features(x)
        x=self.classifier(x)
        return x

In [23]:
# Transfer learning with ResNet
from torchvision import models
class ResNET(nn.Module):
    def __init__(self):
        super(ResNET,self).__init__()
        self.resize=nn.Upsample(
            size=(128,128),
            mode="bilinear",
            align_corners=False
        )
        self.resnet=models.resnet18(
            weights=models.ResNet18_Weights.DEFAULT
        )
        self.resnet.fc=nn.Linear(self.resnet.fc.in_features,10)

    def forward(self,x):
        x=self.resize(x)
        x=self.resnet(x)
        return x

In [14]:
cnn=CNN()
train_model(cnn,trainloader,10)

In [28]:
resnet=ResNET()
train_model(resnet,trainloader,5)

epoch: 1: 100%|██████████| 1563/1563 [41:33<00:00,  1.60s/it, loss=0.347] 


epoch:1  loss:0.34739541290035936


epoch: 2:   2%|▏         | 30/1563 [00:48<41:13,  1.61s/it, loss=0.0932]


KeyboardInterrupt: 

In [29]:
print(f"Accuracy for custom CNN model:{eval_model(cnn,testloader):.2f}%")
print(f"Accuracy for ResNet model:{eval_model(resnet,testloader):.2f}%")

100%|██████████| 313/313 [00:31<00:00,  9.81it/s]


Accuracy for custom CNN model:80.57%


100%|██████████| 313/313 [02:44<00:00,  1.90it/s]

Accuracy for ResNet model:93.43%
